# Medical Necessity — Source Data Validation

A step by step walkthrough of `temp_tnet_tripmaster`. Every cell runs one check, prints the
query it ran, and ends with a verdict.

There are no helper functions. Each cell is readable on its own and can be run, edited, or
copied out without tracing anything defined elsewhere.

Verdicts are one of three values:

- **PASS** — the check is clean and nothing downstream is at risk
- **CHECK** — the result needs a human to look at it before it is relied on
- **FAIL** — a number produced downstream will be wrong, not just imprecise

The last cell collects every verdict in one table.

## 1. Settings

In [ ]:
import re
import json

SOURCE_TABLE = "`prod-sandbox`.vivekkumar_patel.temp_tnet_tripmaster"
KNOWLEDGE_PATH = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/med_nec_knowledge.json"

UPSTREAM_TABLES = ["prod.silver_transbroker.triprequest", "prod.silver_transbroker.tripleg"]

EXCLUDED_SERVICE_CODES = ["EMG", "WC", "FWQUOTE", "ORGAN"]
MIN_TEXT_CHARS = 20
TOP_N = 30

VERDICTS = []

print("source table:", SOURCE_TABLE)
print("knowledge file:", KNOWLEDGE_PATH)
print("service codes to exclude:", EXCLUDED_SERVICE_CODES)
print("minimum justification length:", MIN_TEXT_CHARS)

## 2. Does the table exist and how many rows

The first number everything else is measured against.

In [ ]:
query = f"SELECT COUNT(*) AS row_count FROM {SOURCE_TABLE}"
print(query)
print()

ROW_COUNT = spark.sql(query).collect()[0]["row_count"]
print("rows:", f"{ROW_COUNT:,}")
print()

if ROW_COUNT == 0:
    print("VERDICT: FAIL - the table is empty, nothing below will run")
    VERDICTS.append({"step": "2 row count", "verdict": "FAIL", "detail": "table is empty"})
else:
    print("VERDICT: PASS - table is readable and populated")
    VERDICTS.append({"step": "2 row count", "verdict": "PASS",
                     "detail": f"{ROW_COUNT:,} rows"})

## 3. What columns exist

Names and declared types. Read this before assuming any column contains what its name
suggests.

In [ ]:
query = f"DESCRIBE {SOURCE_TABLE}"
print(query)
print()

schema_df = spark.sql(query).toPandas()
schema_df = schema_df[~schema_df["col_name"].str.startswith("#", na=False)]
schema_df = schema_df[schema_df["col_name"].str.strip() != ""]

print(schema_df[["col_name", "data_type"]].to_string(index=False))
print()

ALL_COLUMNS = schema_df["col_name"].tolist()
STRING_COLUMNS = schema_df[schema_df["data_type"].str.contains("string", case=False, na=False)]["col_name"].tolist()

print("total columns:", len(ALL_COLUMNS))
print("string columns:", len(STRING_COLUMNS))
print()
print("VERDICT: PASS - schema read")
VERDICTS.append({"step": "3 schema", "verdict": "PASS",
                 "detail": f"{len(ALL_COLUMNS)} columns, {len(STRING_COLUMNS)} string"})

## 4. How was this table built

`temp_tnet_tripmaster` is a prepared extract, not a raw system table. This prints the
statement that created it. If it is a view, the view text shows the joins and filters that
define the population.

In [ ]:
query = f"SHOW CREATE TABLE {SOURCE_TABLE}"
print(query)
print()

CREATE_STATEMENT = ""
try:
    CREATE_STATEMENT = spark.sql(query).collect()[0][0]
    print(CREATE_STATEMENT)
    print()
    print("VERDICT: PASS - definition retrieved, read it before trusting the column names")
    VERDICTS.append({"step": "4 table definition", "verdict": "PASS",
                     "detail": "definition retrieved"})
except Exception as e:
    print("could not retrieve:", str(e)[:300])
    print()
    print("VERDICT: CHECK - no definition available, ask Vivek how the extract is built")
    VERDICTS.append({"step": "4 table definition", "verdict": "CHECK",
                     "detail": "definition not retrievable"})

## 5. Who owns it and when was it last written

Type tells you whether it is a managed table, an external table, or a view. Created time and
owner tell you who to ask when something looks wrong.

In [ ]:
query = f"DESCRIBE EXTENDED {SOURCE_TABLE}"
print(query)
print()

detail_df = spark.sql(query).toPandas()
wanted = ["Type", "Provider", "Owner", "Created Time", "Created By", "Last Access",
          "Location", "Comment", "View Text", "Table Properties"]
meta_df = detail_df[detail_df["col_name"].isin(wanted)]

print(meta_df[["col_name", "data_type"]].to_string(index=False))
print()

table_type = ""
rows = meta_df[meta_df["col_name"] == "Type"]["data_type"].tolist()
if rows:
    table_type = rows[0]
print("table type:", table_type if table_type else "unknown")
print()
print("VERDICT: PASS - metadata retrieved")
VERDICTS.append({"step": "5 table metadata", "verdict": "PASS",
                 "detail": f"type {table_type}" if table_type else "type unknown"})

## 6. When was the data last refreshed

The last ten write operations. A table that has not been written in months is a stale snapshot,
and any conclusion drawn from it describes the past rather than the present.

In [ ]:
query = f"DESCRIBE HISTORY {SOURCE_TABLE}"
print(query)
print()

try:
    history_df = (spark.sql(query)
                  .select("version", "timestamp", "userName", "operation")
                  .orderBy("version", ascending=False)
                  .limit(10)
                  .toPandas())
    print(history_df.to_string(index=False))
    print()
    last_write = history_df["timestamp"].max()
    print("most recent write:", last_write)
    print()
    print("VERDICT: PASS - write history available, confirm the date is recent enough")
    VERDICTS.append({"step": "6 refresh history", "verdict": "PASS",
                     "detail": f"last write {last_write}"})
except Exception as e:
    print("could not retrieve:", str(e)[:300])
    print()
    print("VERDICT: CHECK - no history, the refresh date cannot be confirmed from here")
    VERDICTS.append({"step": "6 refresh history", "verdict": "CHECK",
                     "detail": "history not available"})

## 7. Are the upstream tables reachable

The extract is believed to come from the trip request and trip leg tables. If the upstream row
counts are far above the extract, the extract is filtered, and that filter is part of the
analysis whether or not anyone documented it.

In [ ]:
upstream_rows = []
for table_name in UPSTREAM_TABLES:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    print(query)
    try:
        n = spark.sql(query).collect()[0]["row_count"]
        print("  rows:", f"{n:,}")
        upstream_rows.append({"table": table_name, "rows": n, "reachable": True})
    except Exception as e:
        print("  not reachable:", str(e)[:150])
        upstream_rows.append({"table": table_name, "rows": None, "reachable": False})
    print()

upstream_df = pd.DataFrame(upstream_rows)
print(upstream_df.to_string(index=False))
print()

reachable = [r for r in upstream_rows if r["reachable"]]
if not reachable:
    print("VERDICT: CHECK - no upstream table reachable, lineage cannot be confirmed here")
    VERDICTS.append({"step": "7 upstream", "verdict": "CHECK", "detail": "none reachable"})
else:
    biggest = max(r["rows"] for r in reachable)
    ratio = biggest / max(ROW_COUNT, 1)
    print("largest upstream is", f"{ratio:.1f}", "times the extract")
    print()
    if ratio > 1.5:
        print("VERDICT: CHECK - the extract is a filtered subset, confirm the filter with Vivek")
        VERDICTS.append({"step": "7 upstream", "verdict": "CHECK",
                         "detail": f"upstream is {ratio:.1f}x the extract"})
    else:
        print("VERDICT: PASS - extract size is close to upstream")
        VERDICTS.append({"step": "7 upstream", "verdict": "PASS",
                         "detail": f"upstream is {ratio:.1f}x the extract"})

## 8. Which columns could hold what we need

Every column name is searched for keywords. This does not choose anything. It shows the
candidates so the choice in the next cell is made by a person looking at the list.

In [ ]:
import pandas as pd

KEYWORDS = {
    "order_id": ["order", "trip", "transport", "request"],
    "notes": ["note", "reason", "necessity", "justification", "clinical", "comment", "remark"],
    "facility": ["facility", "origin", "pickup", "sending", "location"],
    "service_level": ["service", "level", "los", "modality"],
    "created": ["created", "date", "time", "ordered"],
    "payer": ["payer", "payor", "insurance", "plan"],
}

for role, words in KEYWORDS.items():
    matches = []
    for col in ALL_COLUMNS:
        low = col.lower()
        for w in words:
            if w in low:
                matches.append(col)
                break
    print("-" * 90)
    print(role, "  keywords:", words)
    print("  candidates:", matches if matches else "NONE FOUND")
print("-" * 90)
print()
print("VERDICT: PASS - candidate lists printed, choose in the next cell")
VERDICTS.append({"step": "8 candidates", "verdict": "PASS", "detail": "candidates listed"})

## 9. Choose the columns

Edit this cell. These six names drive every check below and the review notebook. Getting the
justification column wrong here means the whole analysis reads the wrong field.

In [ ]:
COL_ORDER_ID = ""
COL_NOTES = ""
COL_FACILITY = ""
COL_SERVICE = ""
COL_CREATED = ""
COL_PAYER = ""

chosen = {
    "order_id": COL_ORDER_ID,
    "notes": COL_NOTES,
    "facility": COL_FACILITY,
    "service_level": COL_SERVICE,
    "created": COL_CREATED,
    "payer": COL_PAYER,
}

missing = [role for role, col in chosen.items() if not col]
wrong = [f"{role}={col}" for role, col in chosen.items() if col and col not in ALL_COLUMNS]

for role, col in chosen.items():
    status = "not set" if not col else ("NOT IN TABLE" if col not in ALL_COLUMNS else "ok")
    print(f"{role:15s} {col if col else '(blank)':45s} {status}")
print()

if wrong:
    print("VERDICT: FAIL - these names do not exist in the table:", wrong)
    VERDICTS.append({"step": "9 column choice", "verdict": "FAIL",
                     "detail": "names not in table: " + ", ".join(wrong)})
elif missing:
    print("VERDICT: FAIL - not set:", missing)
    print("Fill these in from the candidate lists in section 8, then re-run from here.")
    VERDICTS.append({"step": "9 column choice", "verdict": "FAIL",
                     "detail": "unset roles: " + ", ".join(missing)})
else:
    print("VERDICT: PASS - all six columns chosen and present in the table")
    VERDICTS.append({"step": "9 column choice", "verdict": "PASS", "detail": "six columns set"})

## 10. Is there one row per order

If the identifier repeats, the table is at leg or event grain. Every rate computed per row is
then weighted toward multi leg trips, and a percentage of orders is not a percentage of
orders.

In [ ]:
query = f"""
SELECT
  COUNT(*) AS rows_total,
  COUNT(DISTINCT `{COL_ORDER_ID}`) AS distinct_ids,
  COUNT(*) - COUNT(`{COL_ORDER_ID}`) AS null_ids
FROM {SOURCE_TABLE}
"""
print(query)

grain_df = spark.sql(query).toPandas()
print(grain_df.to_string(index=False))
print()

rows_total = int(grain_df["rows_total"][0])
distinct_ids = int(grain_df["distinct_ids"][0])
null_ids = int(grain_df["null_ids"][0])
rows_per_id = rows_total / max(distinct_ids, 1)

print("rows per identifier:", round(rows_per_id, 3))
print("identifiers that are null:", f"{null_ids:,}")
print()

if rows_per_id > 1.0:
    query2 = f"""
SELECT `{COL_ORDER_ID}` AS order_id, COUNT(*) AS times_appearing
FROM {SOURCE_TABLE}
GROUP BY `{COL_ORDER_ID}`
HAVING COUNT(*) > 1
ORDER BY times_appearing DESC
LIMIT 10
"""
    print(query2)
    print(spark.sql(query2).toPandas().to_string(index=False))
    print()
    print("VERDICT: FAIL - not one row per order, deduplicate before computing any rate")
    VERDICTS.append({"step": "10 grain", "verdict": "FAIL",
                     "detail": f"{round(rows_per_id, 3)} rows per identifier"})
else:
    print("VERDICT: PASS - one row per order")
    VERDICTS.append({"step": "10 grain", "verdict": "PASS", "detail": "unique identifier"})

## 11. Are there duplicate rows

Exact duplicates inflate every count without changing any distribution, so they are invisible
in a chart and wrong in a total.

In [ ]:
query = f"""
SELECT COUNT(*) AS distinct_rows
FROM (SELECT DISTINCT * FROM {SOURCE_TABLE})
"""
print(query)
print()

distinct_rows = spark.sql(query).collect()[0]["distinct_rows"]
duplicate_rows = ROW_COUNT - distinct_rows

print("rows in table:      ", f"{ROW_COUNT:,}")
print("distinct rows:      ", f"{distinct_rows:,}")
print("exact duplicates:   ", f"{duplicate_rows:,}")
print()

if duplicate_rows > 0:
    print("VERDICT: CHECK -", f"{duplicate_rows:,}",
          "rows are exact copies, confirm this is expected")
    VERDICTS.append({"step": "11 duplicates", "verdict": "CHECK",
                     "detail": f"{duplicate_rows:,} exact duplicate rows"})
else:
    print("VERDICT: PASS - no exact duplicate rows")
    VERDICTS.append({"step": "11 duplicates", "verdict": "PASS", "detail": "none"})

## 12. How complete are the six columns we chose

Null and blank are counted separately. A column filled with empty strings passes a null check
and fails everything after it.

In [ ]:
query = f"""
SELECT
  SUM(CASE WHEN `{COL_ORDER_ID}` IS NULL THEN 1 ELSE 0 END) AS order_id_null,
  SUM(CASE WHEN `{COL_NOTES}` IS NULL THEN 1 ELSE 0 END) AS notes_null,
  SUM(CASE WHEN `{COL_NOTES}` IS NOT NULL AND length(trim(`{COL_NOTES}`)) = 0 THEN 1 ELSE 0 END) AS notes_blank,
  SUM(CASE WHEN `{COL_FACILITY}` IS NULL THEN 1 ELSE 0 END) AS facility_null,
  SUM(CASE WHEN `{COL_SERVICE}` IS NULL THEN 1 ELSE 0 END) AS service_null,
  SUM(CASE WHEN `{COL_CREATED}` IS NULL THEN 1 ELSE 0 END) AS created_null,
  SUM(CASE WHEN `{COL_PAYER}` IS NULL THEN 1 ELSE 0 END) AS payer_null
FROM {SOURCE_TABLE}
"""
print(query)

completeness_df = spark.sql(query).toPandas()
print(completeness_df.to_string(index=False))
print()

notes_missing = int(completeness_df["notes_null"][0]) + int(completeness_df["notes_blank"][0])
notes_present_pct = (ROW_COUNT - notes_missing) / max(ROW_COUNT, 1) * 100

print("justification present on:", f"{ROW_COUNT - notes_missing:,}", "rows",
      f"({notes_present_pct:.1f}%)")
print()

if notes_present_pct < 50:
    print("VERDICT: FAIL - fewer than half the orders have any justification text")
    VERDICTS.append({"step": "12 completeness", "verdict": "FAIL",
                     "detail": f"justification present on {notes_present_pct:.1f}% of rows"})
elif notes_present_pct < 90:
    print("VERDICT: CHECK - a meaningful share of orders have no justification text")
    VERDICTS.append({"step": "12 completeness", "verdict": "CHECK",
                     "detail": f"justification present on {notes_present_pct:.1f}% of rows"})
else:
    print("VERDICT: PASS - justification is populated on nearly every row")
    VERDICTS.append({"step": "12 completeness", "verdict": "PASS",
                     "detail": f"justification present on {notes_present_pct:.1f}% of rows"})

## 13. What period does the data cover

Confirms the window the analysis actually describes.

In [ ]:
query = f"""
SELECT
  MIN(`{COL_CREATED}`) AS earliest,
  MAX(`{COL_CREATED}`) AS latest,
  SUM(CASE WHEN `{COL_CREATED}` IS NULL THEN 1 ELSE 0 END) AS no_date
FROM {SOURCE_TABLE}
"""
print(query)

dates_df = spark.sql(query).toPandas()
print(dates_df.to_string(index=False))
print()

no_date = int(dates_df["no_date"][0])
print("rows with no date:", f"{no_date:,}")
print()

if no_date > ROW_COUNT * 0.05:
    print("VERDICT: CHECK - more than 5 percent of rows have no date")
    VERDICTS.append({"step": "13 date range", "verdict": "CHECK",
                     "detail": f"{no_date:,} rows with no date"})
else:
    print("VERDICT: PASS - date column is populated and bounded")
    VERDICTS.append({"step": "13 date range", "verdict": "PASS",
                     "detail": f"{dates_df['earliest'][0]} to {dates_df['latest'][0]}"})

## 14. Is volume steady across the period

A month at a fraction of the others usually means a partial load rather than a real decline in
transports.

In [ ]:
query = f"""
SELECT
  date_format(`{COL_CREATED}`, 'yyyy-MM') AS month,
  COUNT(*) AS orders
FROM {SOURCE_TABLE}
WHERE `{COL_CREATED}` IS NOT NULL
GROUP BY date_format(`{COL_CREATED}`, 'yyyy-MM')
ORDER BY month
"""
print(query)

monthly_df = spark.sql(query).toPandas()
monthly_df["pct_of_total"] = (monthly_df["orders"] / max(ROW_COUNT, 1) * 100).round(2)
print(monthly_df.to_string(index=False))
print()

median_month = monthly_df["orders"].median()
thin_months = monthly_df[monthly_df["orders"] < median_month * 0.5]["month"].tolist()

print("median month:", f"{int(median_month):,}", "orders")
print("months below half the median:", thin_months if thin_months else "none")
print()

if thin_months:
    print("VERDICT: CHECK - these months look like partial loads, not real volume")
    VERDICTS.append({"step": "14 monthly volume", "verdict": "CHECK",
                     "detail": "thin months: " + ", ".join(thin_months)})
else:
    print("VERDICT: PASS - volume is steady across the period")
    VERDICTS.append({"step": "14 monthly volume", "verdict": "PASS",
                     "detail": f"{len(monthly_df)} months, no thin months"})

## 15. What values does the service level column hold

The codes being excluded downstream have to actually appear here, spelled the way the filter
expects.

In [ ]:
query = f"""
SELECT `{COL_SERVICE}` AS service_level, COUNT(*) AS orders
FROM {SOURCE_TABLE}
GROUP BY `{COL_SERVICE}`
ORDER BY orders DESC
LIMIT {TOP_N}
"""
print(query)

service_df = spark.sql(query).toPandas()
service_df["pct"] = (service_df["orders"] / max(ROW_COUNT, 1) * 100).round(2)
print(service_df.to_string(index=False))
print()

present_values = [str(v).upper() for v in service_df["service_level"].tolist() if v is not None]
found_codes = []
not_found_codes = []
for code in EXCLUDED_SERVICE_CODES:
    hit = False
    for v in present_values:
        if code in v:
            hit = True
            break
    if hit:
        found_codes.append(code)
    else:
        not_found_codes.append(code)

print("codes to exclude that appear here:    ", found_codes if found_codes else "none")
print("codes to exclude that do NOT appear:  ", not_found_codes if not_found_codes else "none")
print()

if not_found_codes:
    print("VERDICT: CHECK - these codes are never used, the filter for them does nothing:",
          not_found_codes)
    VERDICTS.append({"step": "15 service values", "verdict": "CHECK",
                     "detail": "unused exclusion codes: " + ", ".join(not_found_codes)})
else:
    print("VERDICT: PASS - every exclusion code appears in the data")
    VERDICTS.append({"step": "15 service values", "verdict": "PASS",
                     "detail": f"{len(service_df)} distinct service values"})

## 16. Which columns are categories rather than free text

A string column with few distinct values is a category list, no matter what its name suggests.
Read its values instead of parsing it.

In [ ]:
cardinality_rows = []
for col in STRING_COLUMNS:
    q = f"SELECT COUNT(DISTINCT `{col}`) AS d FROM {SOURCE_TABLE}"
    d = spark.sql(q).collect()[0]["d"]
    cardinality_rows.append({"column": col, "distinct_values": int(d)})

cardinality_df = pd.DataFrame(cardinality_rows).sort_values("distinct_values").reset_index(drop=True)
cardinality_df["share_of_rows"] = (cardinality_df["distinct_values"] / max(ROW_COUNT, 1)).round(5)

shapes = []
for d in cardinality_df["distinct_values"]:
    if d <= 60:
        shapes.append("category list")
    elif d / max(ROW_COUNT, 1) > 0.5:
        shapes.append("free text or identifier")
    else:
        shapes.append("high cardinality")
cardinality_df["reads_as"] = shapes

print(cardinality_df.to_string(index=False))
print()

CATEGORY_COLUMNS = cardinality_df[cardinality_df["reads_as"] == "category list"]["column"].tolist()
TEXT_COLUMNS = cardinality_df[cardinality_df["reads_as"] != "category list"]["column"].tolist()

print("category columns:", len(CATEGORY_COLUMNS))
print("free text or high cardinality columns:", len(TEXT_COLUMNS))
print()
print("VERDICT: PASS - column shapes established")
VERDICTS.append({"step": "16 cardinality", "verdict": "PASS",
                 "detail": f"{len(CATEGORY_COLUMNS)} category columns"})

## 17. Would any column be unsafe to parse as a number

This is the check that would have caught the nurse navigation failure. A category column whose
values contain digits reads like a score and behaves like a label. A value containing a range
such as `1-5` will hand a parser the first digit and look correct forever.

In [ ]:
risk_rows = []
value_rows = []

for col in CATEGORY_COLUMNS:
    q = f"""
SELECT `{col}` AS value, COUNT(*) AS orders
FROM {SOURCE_TABLE}
GROUP BY `{col}`
ORDER BY orders DESC
LIMIT {TOP_N}
"""
    vals_df = spark.sql(q).toPandas()
    vals_df["column"] = col
    value_rows.append(vals_df[["column", "value", "orders"]])

    values = [str(v) for v in vals_df["value"].tolist() if v is not None]
    with_digits = 0
    with_ranges = 0
    for v in values:
        if re.search(r"[A-Za-z].*\d|\d.*[A-Za-z]", v):
            with_digits += 1
        if re.search(r"\d\s*[-\u2013]\s*\d", v):
            with_ranges += 1

    if with_ranges > 0:
        risk = "HIGH"
    elif with_digits > 0:
        risk = "MEDIUM"
    else:
        risk = "low"

    risk_rows.append({"column": col, "values_with_digits": with_digits,
                      "values_with_ranges": with_ranges, "parse_risk": risk})

risk_df = pd.DataFrame(risk_rows).sort_values("parse_risk").reset_index(drop=True)
values_df = pd.concat(value_rows, ignore_index=True) if value_rows else pd.DataFrame()

print(risk_df.to_string(index=False))
print()

high_risk = risk_df[risk_df["parse_risk"] == "HIGH"]["column"].tolist()
medium_risk = risk_df[risk_df["parse_risk"] == "MEDIUM"]["column"].tolist()

for col in high_risk + medium_risk:
    print("-" * 90)
    print(col)
    print(values_df[values_df["column"] == col][["value", "orders"]].to_string(index=False))
print("-" * 90)
print()

if high_risk:
    print("VERDICT: FAIL - these columns contain ranges, never parse a number from them:",
          high_risk)
    VERDICTS.append({"step": "17 parse risk", "verdict": "FAIL",
                     "detail": "ranges in: " + ", ".join(high_risk)})
elif medium_risk:
    print("VERDICT: CHECK - these columns mix letters and digits, read the values first:",
          medium_risk)
    VERDICTS.append({"step": "17 parse risk", "verdict": "CHECK",
                     "detail": "digits in labels: " + ", ".join(medium_risk)})
else:
    print("VERDICT: PASS - no category column looks numeric")
    VERDICTS.append({"step": "17 parse risk", "verdict": "PASS", "detail": "none"})

## 18. Load the medical necessity criteria

The criteria file is the same one the review notebook uses. It is loaded here so the checks
below measure the data against the actual criteria rather than against a guess.

In [ ]:
with open(KNOWLEDGE_PATH, "r") as f:
    KB = json.load(f)

ALL_TERMS = []
for c in KB["concepts"]:
    ALL_TERMS.extend(c["terms"])
for e in KB["bed_confinement"]["elements"]:
    ALL_TERMS.extend(e["terms"])
for e in KB["exclusions"]:
    ALL_TERMS.extend(e["terms"])
ALL_TERMS = sorted(set(ALL_TERMS))

UNSAFE = []
for t in ALL_TERMS + KB["non_specific_phrases"]:
    if re.search(r"[.^$*+?()\[\]{}|\\']", t):
        UNSAFE.append(t)

ALL_TERMS_PATTERN = "|".join(t for t in sorted(ALL_TERMS, key=len, reverse=True))
ALL_TERMS_RLIKE = f"(^|[^a-z0-9])({ALL_TERMS_PATTERN})([^a-z0-9]|$)"

VAGUE_PATTERN = "|".join(t for t in
                        sorted(KB["non_specific_phrases"], key=len, reverse=True))
VAGUE_RLIKE = f"(^|[^a-z0-9])({VAGUE_PATTERN})([^a-z0-9]|$)"

print("knowledge version:", KB["knowledge_version"])
print("concepts:", len(KB["concepts"]))
print("bed confinement elements:", len(KB["bed_confinement"]["elements"]))
print("exclusions:", len(KB["exclusions"]))
print("distinct terms across all of them:", len(ALL_TERMS))
print()
print("terms containing characters unsafe inside a SQL pattern:", UNSAFE if UNSAFE else "none")
print()

if UNSAFE:
    print("VERDICT: FAIL - these terms contain regex or quote characters and will break the")
    print("queries below. Reword them in med_nec_knowledge.json.")
    VERDICTS.append({"step": "18 criteria", "verdict": "FAIL",
                     "detail": "unsafe terms: " + ", ".join(UNSAFE[:5])})
else:
    print("VERDICT: PASS - criteria loaded, all terms safe to embed in a query")
    VERDICTS.append({"step": "18 criteria", "verdict": "PASS",
                     "detail": f"{len(ALL_TERMS)} terms across {len(KB['concepts'])} concepts"})

## 19. Is the chosen column really the justification

Name matching is a guess. This tests every free text column against the criteria vocabulary.
The column that mentions clinical criteria most often is the justification, whatever it is
called.

In [ ]:
proof_rows = []
for col in TEXT_COLUMNS:
    q = f"""
SELECT
  COUNT(*) AS rows_total,
  SUM(CASE WHEN `{col}` IS NOT NULL AND length(trim(`{col}`)) >= {MIN_TEXT_CHARS} THEN 1 ELSE 0 END) AS usable,
  SUM(CASE WHEN lower(`{col}`) rlike '{ALL_TERMS_RLIKE}' THEN 1 ELSE 0 END) AS mentions_criteria,
  CAST(AVG(length(`{col}`)) AS INT) AS avg_chars
FROM {SOURCE_TABLE}
"""
    r = spark.sql(q).collect()[0]
    proof_rows.append({
        "column": col,
        "usable_pct": round(r["usable"] / max(ROW_COUNT, 1) * 100, 1),
        "mentions_criteria_pct": round(r["mentions_criteria"] / max(ROW_COUNT, 1) * 100, 1),
        "avg_chars": r["avg_chars"] if r["avg_chars"] else 0,
    })

proof_df = (pd.DataFrame(proof_rows)
            .sort_values("mentions_criteria_pct", ascending=False)
            .reset_index(drop=True))
print(proof_df.to_string(index=False))
print()

best_column = proof_df["column"][0]
best_rate = proof_df["mentions_criteria_pct"][0]
chosen_row = proof_df[proof_df["column"] == COL_NOTES]
chosen_rate = float(chosen_row["mentions_criteria_pct"].iloc[0]) if len(chosen_row) else 0.0

print("column chosen in section 9:", COL_NOTES, "  mentions criteria on", chosen_rate, "percent")
print("highest scoring column:    ", best_column, "  mentions criteria on", best_rate, "percent")
print()

if best_column != COL_NOTES:
    print("VERDICT: FAIL - a different column mentions clinical criteria more often.")
    print("Read both before continuing. Reviewing the wrong column wastes every model call.")
    VERDICTS.append({"step": "19 justification column", "verdict": "FAIL",
                     "detail": f"chose {COL_NOTES} at {chosen_rate}%, "
                               f"{best_column} scores {best_rate}%"})
else:
    print("VERDICT: PASS - the chosen column is the one that mentions criteria most often")
    VERDICTS.append({"step": "19 justification column", "verdict": "PASS",
                     "detail": f"{COL_NOTES} at {chosen_rate}%"})

## 20. How long is the justification text

Short text cannot describe a patient's condition. This is the ceiling on what any review, human
or model, can conclude.

In [ ]:
query = f"""
SELECT
  SUM(CASE WHEN `{COL_NOTES}` IS NULL OR length(trim(`{COL_NOTES}`)) = 0 THEN 1 ELSE 0 END) AS empty,
  SUM(CASE WHEN length(trim(`{COL_NOTES}`)) BETWEEN 1 AND {MIN_TEXT_CHARS - 1} THEN 1 ELSE 0 END) AS too_short,
  SUM(CASE WHEN length(trim(`{COL_NOTES}`)) >= {MIN_TEXT_CHARS} THEN 1 ELSE 0 END) AS usable,
  CAST(percentile_approx(length(trim(`{COL_NOTES}`)), 0.25) AS INT) AS p25_chars,
  CAST(percentile_approx(length(trim(`{COL_NOTES}`)), 0.50) AS INT) AS median_chars,
  CAST(percentile_approx(length(trim(`{COL_NOTES}`)), 0.95) AS INT) AS p95_chars,
  MAX(length(trim(`{COL_NOTES}`))) AS max_chars
FROM {SOURCE_TABLE}
"""
print(query)

length_df = spark.sql(query).toPandas()
print(length_df.to_string(index=False))
print()

usable = int(length_df["usable"][0])
usable_pct = usable / max(ROW_COUNT, 1) * 100
median_chars = int(length_df["median_chars"][0])

print("usable justifications:", f"{usable:,}", f"({usable_pct:.1f}%)")
print("median length:", median_chars, "characters")
print()

if usable_pct < 80:
    print("VERDICT: CHECK - a large share of orders carry too little text to review")
    VERDICTS.append({"step": "20 text length", "verdict": "CHECK",
                     "detail": f"{usable_pct:.1f}% usable, median {median_chars} chars"})
else:
    print("VERDICT: PASS - most orders carry enough text to review")
    VERDICTS.append({"step": "20 text length", "verdict": "PASS",
                     "detail": f"{usable_pct:.1f}% usable, median {median_chars} chars"})

## 21. Is the justification text boilerplate

A string repeated verbatim across hundreds of orders is a template default, not documentation
of a patient's condition. It will score identically every time and it tells a reviewer
nothing.

In [ ]:
query = f"""
SELECT
  lower(trim(`{COL_NOTES}`)) AS justification,
  COUNT(*) AS orders
FROM {SOURCE_TABLE}
WHERE `{COL_NOTES}` IS NOT NULL
  AND length(trim(`{COL_NOTES}`)) >= {MIN_TEXT_CHARS}
GROUP BY lower(trim(`{COL_NOTES}`))
ORDER BY orders DESC
LIMIT {TOP_N}
"""
print(query)

repeated_df = spark.sql(query).toPandas()
repeated_df["chars"] = repeated_df["justification"].str.len()
repeated_df["pct_of_usable"] = (repeated_df["orders"] / max(usable, 1) * 100).round(2)

print(repeated_df[["orders", "pct_of_usable", "chars"]].to_string(index=False))
print()
print("Set SHOW_TEXT to True to print the strings themselves. The source table carries patient")
print("identifiers, so they are withheld by default.")
SHOW_TEXT = False
if SHOW_TEXT:
    print(repeated_df[["orders", "justification"]].to_string(index=False))
print()

top_share = float(repeated_df["pct_of_usable"][0]) if len(repeated_df) else 0.0
top_30_share = float(repeated_df["pct_of_usable"].sum()) if len(repeated_df) else 0.0

print("most common single string covers:", top_share, "percent of usable orders")
print("top", TOP_N, "strings cover:", round(top_30_share, 2), "percent of usable orders")
print()

if top_30_share > 50:
    print("VERDICT: CHECK - over half the justifications are one of", TOP_N, "repeated strings.")
    print("Documentation is largely templated, which limits what any review can distinguish.")
    VERDICTS.append({"step": "21 boilerplate", "verdict": "CHECK",
                     "detail": f"top {TOP_N} strings cover {round(top_30_share, 1)}% of usable"})
else:
    print("VERDICT: PASS - justification text is mostly written per order")
    VERDICTS.append({"step": "21 boilerplate", "verdict": "PASS",
                     "detail": f"top {TOP_N} strings cover {round(top_30_share, 1)}% of usable"})

## 22. How many orders say nothing specific

Text made entirely of phrases like "per MD order" or "needs transport" is populated but empty
of clinical content.

In [ ]:
query = f"""
SELECT
  SUM(CASE WHEN lower(`{COL_NOTES}`) rlike '{ALL_TERMS_RLIKE}' THEN 1 ELSE 0 END) AS mentions_criteria,
  SUM(CASE WHEN lower(`{COL_NOTES}`) rlike '{VAGUE_RLIKE}'
            AND NOT lower(`{COL_NOTES}`) rlike '{ALL_TERMS_RLIKE}' THEN 1 ELSE 0 END) AS vague_only,
  SUM(CASE WHEN NOT lower(`{COL_NOTES}`) rlike '{VAGUE_RLIKE}'
            AND NOT lower(`{COL_NOTES}`) rlike '{ALL_TERMS_RLIKE}' THEN 1 ELSE 0 END) AS neither
FROM {SOURCE_TABLE}
WHERE `{COL_NOTES}` IS NOT NULL
  AND length(trim(`{COL_NOTES}`)) >= {MIN_TEXT_CHARS}
"""
print(query)

content_df = spark.sql(query).toPandas()
print(content_df.to_string(index=False))
print()

mentions = int(content_df["mentions_criteria"][0])
vague_only = int(content_df["vague_only"][0])
neither = int(content_df["neither"][0])

print("mentions at least one criterion:", f"{mentions:,}",
      f"({mentions / max(usable, 1) * 100:.1f}%)")
print("only non-specific phrasing:     ", f"{vague_only:,}",
      f"({vague_only / max(usable, 1) * 100:.1f}%)")
print("neither, unrecognised wording:  ", f"{neither:,}",
      f"({neither / max(usable, 1) * 100:.1f}%)")
print()

if neither / max(usable, 1) > 0.4:
    print("VERDICT: CHECK - a large share of text matches nothing in the criteria file.")
    print("Read a sample. Either these facilities write differently or the vocabulary has gaps.")
    VERDICTS.append({"step": "22 content", "verdict": "CHECK",
                     "detail": f"{neither / max(usable, 1) * 100:.1f}% matches nothing"})
else:
    print("VERDICT: PASS - most justification text is recognisable to the criteria")
    VERDICTS.append({"step": "22 content", "verdict": "PASS",
                     "detail": f"{mentions / max(usable, 1) * 100:.1f}% mentions a criterion"})

## 23. How many orders are actually in scope

Each filter runs as its own query and the rows it removes are recorded. The last number here is
the denominator for every medical necessity rate. It is not the table row count.

In [ ]:
funnel_rows = [{"step": "all rows in table", "orders": ROW_COUNT, "removed": 0}]
where_parts = []
previous = ROW_COUNT

for code in EXCLUDED_SERVICE_CODES:
    where_parts.append(f"upper(`{COL_SERVICE}`) NOT RLIKE '(^|[^A-Z0-9]){code}([^A-Z0-9]|$)'")
    where_clause = "\n  AND ".join(where_parts)
    q = f"SELECT COUNT(*) AS n\nFROM {SOURCE_TABLE}\nWHERE {where_clause}"
    print("-" * 90)
    print(f"remove service code {code}")
    print(q)
    n = spark.sql(q).collect()[0]["n"]
    print("remaining:", f"{n:,}", "  removed by this step:", f"{previous - n:,}")
    funnel_rows.append({"step": f"remove service code {code}", "orders": int(n),
                        "removed": int(previous - n)})
    previous = n

where_parts.append(f"`{COL_NOTES}` IS NOT NULL")
where_clause = "\n  AND ".join(where_parts)
q = f"SELECT COUNT(*) AS n\nFROM {SOURCE_TABLE}\nWHERE {where_clause}"
print("-" * 90)
print("require a justification")
print(q)
n = spark.sql(q).collect()[0]["n"]
print("remaining:", f"{n:,}", "  removed by this step:", f"{previous - n:,}")
funnel_rows.append({"step": "require a justification", "orders": int(n),
                    "removed": int(previous - n)})
previous = n

where_parts.append(f"length(trim(`{COL_NOTES}`)) >= {MIN_TEXT_CHARS}")
where_clause = "\n  AND ".join(where_parts)
q = f"SELECT COUNT(*) AS n\nFROM {SOURCE_TABLE}\nWHERE {where_clause}"
print("-" * 90)
print(f"justification at least {MIN_TEXT_CHARS} characters")
print(q)
n = spark.sql(q).collect()[0]["n"]
print("remaining:", f"{n:,}", "  removed by this step:", f"{previous - n:,}")
funnel_rows.append({"step": f"justification at least {MIN_TEXT_CHARS} characters",
                    "orders": int(n), "removed": int(previous - n)})
previous = n

print("-" * 90)
print()
funnel_df = pd.DataFrame(funnel_rows)
funnel_df["pct_of_table"] = (funnel_df["orders"] / max(ROW_COUNT, 1) * 100).round(2)
print(funnel_df.to_string(index=False))
print()

IN_SCOPE = previous
in_scope_pct = IN_SCOPE / max(ROW_COUNT, 1) * 100
print("ORDERS IN SCOPE:", f"{IN_SCOPE:,}", f"({in_scope_pct:.1f}% of the table)")
print()

if in_scope_pct < 25:
    print("VERDICT: CHECK - the filters remove most of the table, confirm they are correct")
    VERDICTS.append({"step": "23 scope", "verdict": "CHECK",
                     "detail": f"{IN_SCOPE:,} in scope, {in_scope_pct:.1f}% of table"})
else:
    print("VERDICT: PASS - scope established")
    VERDICTS.append({"step": "23 scope", "verdict": "PASS",
                     "detail": f"{IN_SCOPE:,} in scope, {in_scope_pct:.1f}% of table"})

## 24. The query that pulls the population

The full extract statement. Copy it into a SQL editor or hand it to anyone who needs to
reproduce this population without running the notebook.

In [ ]:
select_clause = ",\n  ".join([f"`{COL_ORDER_ID}`", f"`{COL_CREATED}`", f"`{COL_FACILITY}`",
                                f"`{COL_SERVICE}`", f"`{COL_PAYER}`", f"`{COL_NOTES}`"])
where_clause = "\n  AND ".join(where_parts)

EXTRACT_SQL = f"SELECT\n  {select_clause}\nFROM {SOURCE_TABLE}\nWHERE {where_clause}"

print(EXTRACT_SQL)
print()

check = spark.sql(f"SELECT COUNT(*) AS n FROM ({EXTRACT_SQL})").collect()[0]["n"]
print("rows returned by this query:", f"{check:,}")
print("rows expected from the funnel:", f"{IN_SCOPE:,}")
print()

spark.sql(EXTRACT_SQL).createOrReplaceTempView("med_nec_scope")
print("temporary view created: med_nec_scope")
print()

if check != IN_SCOPE:
    print("VERDICT: FAIL - the extract query does not reproduce the funnel count")
    VERDICTS.append({"step": "24 extract query", "verdict": "FAIL",
                     "detail": f"query returns {check:,}, funnel says {IN_SCOPE:,}"})
else:
    print("VERDICT: PASS - the extract query reproduces the funnel count exactly")
    VERDICTS.append({"step": "24 extract query", "verdict": "PASS",
                     "detail": f"{check:,} rows"})

## 25. Which criteria actually appear in the data

Each concept is counted against the in scope population. A concept at zero either does not
occur in these facilities or is written here in wording the criteria file does not know, and
only reading notes tells you which.

In [ ]:
coverage_rows = []

for c in KB["concepts"]:
    pattern = "|".join(t for t in sorted(c["terms"], key=len, reverse=True))
    q = f"""
SELECT COUNT(*) AS n
FROM med_nec_scope
WHERE lower(`{COL_NOTES}`) rlike '(^|[^a-z0-9])({pattern})([^a-z0-9]|$)'
"""
    n = spark.sql(q).collect()[0]["n"]
    coverage_rows.append({"kind": "concept", "name": c["concept"], "axis": c["axis"],
                          "status": c["status"], "orders": int(n),
                          "pct_of_scope": round(n / max(IN_SCOPE, 1) * 100, 2)})

for e in KB["bed_confinement"]["elements"]:
    pattern = "|".join(t for t in sorted(e["terms"], key=len, reverse=True))
    q = f"""
SELECT COUNT(*) AS n
FROM med_nec_scope
WHERE lower(`{COL_NOTES}`) rlike '(^|[^a-z0-9])({pattern})([^a-z0-9]|$)'
"""
    n = spark.sql(q).collect()[0]["n"]
    coverage_rows.append({"kind": "bed element", "name": e["element"], "axis": "mobility",
                          "status": "cited", "orders": int(n),
                          "pct_of_scope": round(n / max(IN_SCOPE, 1) * 100, 2)})

for e in KB["exclusions"]:
    pattern = "|".join(t for t in sorted(e["terms"], key=len, reverse=True))
    q = f"""
SELECT COUNT(*) AS n
FROM med_nec_scope
WHERE lower(`{COL_NOTES}`) rlike '(^|[^a-z0-9])({pattern})([^a-z0-9]|$)'
"""
    n = spark.sql(q).collect()[0]["n"]
    coverage_rows.append({"kind": "exclusion", "name": e["exclusion_id"], "axis": "",
                          "status": "", "orders": int(n),
                          "pct_of_scope": round(n / max(IN_SCOPE, 1) * 100, 2)})

coverage_df = (pd.DataFrame(coverage_rows)
               .sort_values(["kind", "orders"], ascending=[True, False])
               .reset_index(drop=True))
print(coverage_df.to_string(index=False))
print()

never_seen = coverage_df[coverage_df["orders"] == 0]["name"].tolist()
print("criteria that never appear:", never_seen if never_seen else "none")
print()

if never_seen:
    print("VERDICT: CHECK - read a sample of notes to decide whether these are absent")
    print("from this population or written in wording the criteria file does not recognise.")
    VERDICTS.append({"step": "25 criteria coverage", "verdict": "CHECK",
                     "detail": f"{len(never_seen)} criteria never appear"})
else:
    print("VERDICT: PASS - every criterion appears at least once")
    VERDICTS.append({"step": "25 criteria coverage", "verdict": "PASS",
                     "detail": "all criteria appear"})

## 26. Does bed confinement get documented completely

The training requires all three elements together. This counts how often that actually
happens, which sets a ceiling on how many orders can qualify on the mobility side.

In [ ]:
element_patterns = []
for e in KB["bed_confinement"]["elements"]:
    pattern = "|".join(t for t in sorted(e["terms"], key=len, reverse=True))
    element_patterns.append((e["element"], f"(^|[^a-z0-9])({pattern})([^a-z0-9]|$)"))

case_parts = []
for name, rlike in element_patterns:
    case_parts.append(f"CASE WHEN lower(`{COL_NOTES}`) rlike '{rlike}' THEN 1 ELSE 0 END")
elements_found = " + ".join(case_parts)

query = f"""
SELECT
  {elements_found} AS elements_documented,
  COUNT(*) AS orders
FROM med_nec_scope
GROUP BY {elements_found}
ORDER BY elements_documented
"""
print(query)

bed_df = spark.sql(query).toPandas()
bed_df["pct_of_scope"] = (bed_df["orders"] / max(IN_SCOPE, 1) * 100).round(2)
print(bed_df.to_string(index=False))
print()

complete_rows = bed_df[bed_df["elements_documented"] == 3]["orders"].tolist()
partial_rows = bed_df[bed_df["elements_documented"].isin([1, 2])]["orders"].tolist()
complete = int(complete_rows[0]) if complete_rows else 0
partial = int(sum(partial_rows)) if partial_rows else 0

print("all three elements documented:", f"{complete:,}",
      f"({complete / max(IN_SCOPE, 1) * 100:.1f}%)")
print("one or two elements only:     ", f"{partial:,}",
      f"({partial / max(IN_SCOPE, 1) * 100:.1f}%)")
print()

if partial > complete:
    print("VERDICT: CHECK - incomplete bed confinement claims outnumber complete ones.")
    print("These orders cannot qualify on mobility as written and are the clearest target")
    print("for a documentation prompt at order entry.")
    VERDICTS.append({"step": "26 bed confinement", "verdict": "CHECK",
                     "detail": f"{partial:,} partial vs {complete:,} complete"})
else:
    print("VERDICT: PASS - bed confinement is usually documented completely when claimed")
    VERDICTS.append({"step": "26 bed confinement", "verdict": "PASS",
                     "detail": f"{complete:,} complete, {partial:,} partial"})

## 27. Does documentation quality differ by facility

The business case rests on this split, so it is measured rather than assumed.

In [ ]:
query = f"""
SELECT
  `{COL_FACILITY}` AS facility,
  COUNT(*) AS orders,
  CAST(AVG(length(trim(`{COL_NOTES}`))) AS INT) AS avg_chars,
  ROUND(100.0 * SUM(CASE WHEN lower(`{COL_NOTES}`) rlike '{ALL_TERMS_RLIKE}' THEN 1 ELSE 0 END)
        / COUNT(*), 1) AS mentions_criteria_pct
FROM med_nec_scope
GROUP BY `{COL_FACILITY}`
HAVING COUNT(*) >= 50
ORDER BY orders DESC
LIMIT {TOP_N}
"""
print(query)

facility_df = spark.sql(query).toPandas()
print(facility_df.to_string(index=False))
print()

if len(facility_df) >= 2:
    best = facility_df.sort_values("mentions_criteria_pct", ascending=False).iloc[0]
    worst = facility_df.sort_values("mentions_criteria_pct", ascending=True).iloc[0]
    spread = float(best["mentions_criteria_pct"]) - float(worst["mentions_criteria_pct"])
    print("strongest documentation:", best["facility"], best["mentions_criteria_pct"], "percent")
    print("weakest documentation:  ", worst["facility"], worst["mentions_criteria_pct"], "percent")
    print("spread:", round(spread, 1), "percentage points")
    print()
    if spread > 15:
        print("VERDICT: CHECK - documentation quality varies materially between facilities.")
        print("A single overall rate will hide this. Report by facility.")
        VERDICTS.append({"step": "27 by facility", "verdict": "CHECK",
                         "detail": f"{round(spread, 1)} point spread between facilities"})
    else:
        print("VERDICT: PASS - documentation quality is consistent across facilities")
        VERDICTS.append({"step": "27 by facility", "verdict": "PASS",
                         "detail": f"{round(spread, 1)} point spread"})
else:
    print("VERDICT: CHECK - too few facilities with enough volume to compare")
    VERDICTS.append({"step": "27 by facility", "verdict": "CHECK",
                     "detail": "insufficient facilities"})

## 28. Everything that needs attention

One row per check. Read the FAIL rows first. A FAIL means a number produced downstream will be
wrong rather than imprecise.

In [ ]:
summary_df = pd.DataFrame(VERDICTS)

order_map = {"FAIL": 0, "CHECK": 1, "PASS": 2}
summary_df["sort_key"] = summary_df["verdict"].map(order_map)
summary_df = summary_df.sort_values(["sort_key", "step"]).drop(columns=["sort_key"]).reset_index(drop=True)

print(summary_df.to_string(index=False))
print()
print("=" * 90)
print("checks run: ", len(summary_df))
print("FAIL:       ", int((summary_df["verdict"] == "FAIL").sum()))
print("CHECK:      ", int((summary_df["verdict"] == "CHECK").sum()))
print("PASS:       ", int((summary_df["verdict"] == "PASS").sum()))
print("=" * 90)
print()
print("rows in table:  ", f"{ROW_COUNT:,}")
print("orders in scope:", f"{IN_SCOPE:,}")
print()

fails = summary_df[summary_df["verdict"] == "FAIL"]
if len(fails):
    print("Resolve these before running the review notebook:")
    for r in fails.to_dict("records"):
        print("  -", r["step"], ":", r["detail"])
else:
    print("No blocking failures. The data is fit to run the medical necessity review against.")